# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tkg-create/FlyRank-ML-Track/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding #4 — The Freshness Multiplier

**Claim:** 365+ day content refreshed within 30 days shows a 3.2x health boost and 57x more impressions than similarly old, un-refreshed content. This is framed as *"refresh timing is one of the strongest measured levers available."*

**Where the label comes from:** The freshness-window grouping (`0-30`, `31-90`, ... `361+` days since last update) is observed metadata, not something assigned experimentally. Whether a page got refreshed and when was a human decision, a decision presumably made because the page already looked worth saving.

**Does the validation design carry the claim?** This is a cross-sectional comparison of refreshed vs. un-refreshed pages, not a before/after on the same pages. If editors preferentially refresh pages that already have stronger underlying demand, backlinks, or topical relevance, part of the 3.2x/57x gap belongs to *what got chosen*, not to *refreshing itself*. The paper's own Methodology page flags confounding variables in general, but doesn't apply that caveat to this specific number, and the headline language ("strongest measured levers") reads as causal.

**Constructive framing:** The finding would carry the same practical weight — refresh mature pages that already show demand — while being safer if it read "pages refreshed within 30 days show measured 3.2x/57x gaps over stale peers in this observational comparison" rather than "levers." It's a small wording change but still a notable difference, disclosing that the flagging of that selection into "refreshed" wasn't random without discarding a genuinely useful directional signal.

### Finding #5 — Engagement and Visibility Move Together

**Claim:** "High scroll + high engagement = +11.2 health points." The high_scroll × high engagement bucket scores 50.7 health vs. 39.5 for low_scroll × low. This is presented as evidence that stronger engagement patterns and steadier visibility move together.

**Where the label comes from:** Health Score is a hand-built recipe — impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts), stated on page 5. Scroll depth is one of the four ingredients, worth a fifth of the total score. So "scroll" isn't an independent variable being correlated with health, but rather a direct input to health, by construction.

**Does the validation design carry the claim?:** Only partly, and the paper doesn't separate the two effects. Some fraction of the 11.2-point gap between high_scroll × high and low_scroll × low is mechanical: a page with higher scroll depth earns more scroll-depth points by formula, before any real behavioral pattern is at work. Engagement rate, the other half of the bucket label, isn't a scoring component, so that part of the comparison is a genuine external signal — but the two are bundled into one number, so the finding can't currently say how much of the 11.2 points is "engagement really does track with health" versus "scroll depth is 20% of the score and this bucket has more of it." It's a milder version of the trap on page 27 of the research paper (RF reconstructing Health Score from its own inputs) that was shown in the example, although here it's one input smuggled into a two-variable comparison, not the whole formula.

**Constructive framing:** The finding would be more defensible if split into two claims instead of one: (1) "pages with deeper scroll score higher on Health Score" which is true by construction and therefore not worth reporting as a discovery, and (2) "controlling for scroll depth's direct contribution, engagement rate independently associates with the remaining score" which is the actual new claim. However, it needs the scroll-depth component subtracted out first, or a comparison against a health-score variant computed without scroll depth, before the 11.2-point number can be attributed to "engagement and visibility moving together" rather than partly to the recipe itself.

In [12]:
# Section 1 critiques the published paper, not my own data — no query needed

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The Week 5 model was already already grouped by client_hash_id, so an "honest split" for this audit can't mean introducing grouping for the first time as that's already been done. Instead I'll be checking whether that one grouped split (random_state=42) was a lucky draw or a stable result. The question proposed is "if a different 20% of clients had landed in test, would RF still beat the baseline at K=100/200, or was that one split favorable?"

**Before:** Week 5 evaluated on a single `GroupShuffleSplit` draw (`random_state=42`) — one train/test partition of clients, one number per model per K.

**After:** `GroupKFold(n_splits=5)` grouped by the same `client_hash_id`, same feature set, same model configs, same four K values. Each client's rows land entirely in one fold, never split across folds. This checks whether the Week 5 result depends on which clients ended up in test, not whether grouping matters (that was already decided correctly in Week 5).

The baseline rule isn't refit per fold — it has no parameters to fit, same as Week 5 — it's scored directly on each fold's test rows with the same formula.

RF beating the baseline at K=100/200 was the headline Week 5 finding, so that specific gap will additionally get a bootstrap confidence interval across the 5 fold-level differences: if zero falls inside the interval, the "RF beats baseline" claim doesn't survive stricter scrutiny at that K.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from getpass import getpass

con = duckdb.connect()

hf_token = getpass("Paste your Hugging Face READ token: ")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{base}/fact_content_daily_performance/month=2026-03/data_0.parquet"

Paste your Hugging Face READ token: ··········


In [14]:
import pandas as pd
import numpy as np

# Label — identical to w05/w04
label_df = con.sql(f"""
    WITH halves AS (
        SELECT
            content_hash_id,
            SUM(CASE WHEN report_date < '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_first_half,
            SUM(CASE WHEN report_date >= '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_second_half
        FROM read_parquet('{month_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        content_hash_id,
        CASE WHEN impr_second_half < impr_first_half THEN 1 ELSE 0 END AS is_declining_proxy
    FROM halves
    WHERE impr_first_half > 0
""").df()

# Full-month position/click signal — for the baseline rule's score + tie-break only
pf = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_full,
        SUM(gsc_impressions) AS total_impressions_full,
        SUM(gsc_clicks) AS total_clicks_full
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()
pf_valid = pf.dropna(subset=["avg_position_full"]).copy()
pf_valid["eligible"] = pf_valid["total_impressions_full"] >= 10
pf_valid["zero_clicks_at_position"] = (
    (pf_valid["avg_position_full"] <= 10) & (pf_valid["total_clicks_full"] == 0) & (pf_valid["eligible"])
).astype(int)

# First-half-only features — what the models actually train on
pf_fh = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_fh,
        SUM(gsc_impressions) AS total_impressions_fh,
        SUM(gsc_clicks) AS total_clicks_fh
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE AND report_date < '2026-03-16'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()
pf_fh = pf_fh.dropna(subset=["avg_position_fh"]).copy()
pf_fh["ctr_fh"] = pf_fh["total_clicks_fh"] / pf_fh["total_impressions_fh"]

# Leakage-safe position trend, days 1-15 only
postrend = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN report_date < '2026-03-08' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_wk1,
        AVG(CASE WHEN report_date >= '2026-03-08' AND report_date < '2026-03-16' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_wk2
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE AND report_date < '2026-03-16'
    GROUP BY content_hash_id
""").df()
postrend = postrend.dropna(subset=["avg_position_wk1", "avg_position_wk2"])
postrend["position_change"] = postrend["avg_position_wk2"] - postrend["avg_position_wk1"]
postrend["position_worsened"] = (postrend["position_change"] > 0).astype(int)
postrend_eligible = postrend.merge(pf_valid[["content_hash_id", "eligible"]], on="content_hash_id", how="left")
postrend_eligible = postrend_eligible[postrend_eligible["eligible"] == True].copy()

# Client map for grouping
client_map = con.sql(f"""
    SELECT DISTINCT content_hash_id, client_hash_id
    FROM read_parquet('{month_path}')
""").df()

# Assemble model_df — same joins, same order, as Week 5
model_df = pf_fh.merge(
    pf_valid[["content_hash_id", "total_impressions_full", "eligible", "zero_clicks_at_position"]],
    on="content_hash_id", how="inner"
)
model_df = model_df.merge(
    postrend_eligible[["content_hash_id", "position_change", "position_worsened"]],
    on="content_hash_id", how="left"
)
model_df["has_position_trend"] = model_df["position_change"].notna().astype(int)
model_df["position_change"] = model_df["position_change"].fillna(0)
model_df["position_worsened"] = model_df["position_worsened"].fillna(0).astype(int)
model_df = model_df.merge(client_map, on="content_hash_id", how="left").dropna(subset=["client_hash_id"])
model_df = model_df.merge(label_df, on="content_hash_id", how="inner").sort_values("content_hash_id").reset_index(drop=True)

model_df["log_impressions_fh"] = np.log1p(model_df["total_impressions_fh"])
model_df["log_clicks_fh"] = np.log1p(model_df["total_clicks_fh"])

feature_cols = ["avg_position_fh", "log_impressions_fh", "log_clicks_fh", "ctr_fh",
                "position_change", "has_position_trend"]

print(f"model_df: {model_df.shape}, base rate: {model_df['is_declining_proxy'].mean():.3f}")

# Interaction feature for the diagnostic LR only — the baseline rule's own logic,
# built from first-half-only columns so it's leakage-safe
model_df["eligible_fh"] = model_df["total_impressions_fh"] >= 10
model_df["zero_clicks_at_position_fh"] = (
    (model_df["avg_position_fh"] <= 10)
    & (model_df["total_clicks_fh"] == 0)
    & (model_df["eligible_fh"])
).astype(int)
model_df["zero_clicks_and_worsened_fh"] = (
    model_df["zero_clicks_at_position_fh"] * model_df["position_worsened"]
)

diag_feature_cols = feature_cols + ["zero_clicks_and_worsened_fh"]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

model_df: (150675, 15), base rate: 0.438


In [15]:
# Reproduce the "before" single split
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def precision_at_k(order_labels, k):
    return np.asarray(order_labels)[:k].mean()

def make_lr():
    return Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, random_state=42))])

def make_rf():
    return RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=-1)

def score_baseline(df_slice, y_slice):
    scored = df_slice.copy()
    scored["score"] = scored["zero_clicks_at_position"] * 2 + scored["position_worsened"] * 1
    order = scored.sort_values(["score", "total_impressions_full"], ascending=[False, False]).index
    return y_slice.loc[order].values

X = model_df[feature_cols].astype(float)
y = model_df["is_declining_proxy"].astype(int)
groups = model_df["client_hash_id"]
ks = [20, 50, 100, 200]

# Before: single GroupShuffleSplit draw
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

lr = make_lr(); lr.fit(X_train, y_train)
rf = make_rf(); rf.fit(X_train, y_train)

lr_ranked = y_test.values[np.argsort(-lr.predict_proba(X_test)[:, 1])]
rf_ranked = y_test.values[np.argsort(-rf.predict_proba(X_test)[:, 1])]
baseline_ranked = score_baseline(model_df.loc[X_test.index], y_test)

before_rows = []
for k in ks:
    before_rows.append({"k": k, "baseline_rule": round(precision_at_k(baseline_ranked, k), 3),
                         "logistic_regression": round(precision_at_k(lr_ranked, k), 3),
                         "random_forest": round(precision_at_k(rf_ranked, k), 3)})
before_df = pd.DataFrame(before_rows)
print("BEFORE — single GroupShuffleSplit:")
print(before_df.to_string(index=False))

BEFORE — single GroupShuffleSplit:
  k  baseline_rule  logistic_regression  random_forest
 20           0.70                 0.30          0.600
 50           0.60                 0.20          0.580
100           0.51                 0.27          0.550
200           0.51                 0.33          0.625


In [16]:
# GroupKFold, per-fold results
gkf = GroupKFold(n_splits=5)
fold_records = []

for fold_num, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Diagnostic LR uses the same rows, extra column
    X_train_diag = model_df.loc[X_train.index, diag_feature_cols].astype(float)
    X_test_diag = model_df.loc[X_test.index, diag_feature_cols].astype(float)

    lr = make_lr(); lr.fit(X_train, y_train)
    rf = make_rf(); rf.fit(X_train, y_train)
    lr_diag = make_lr(); lr_diag.fit(X_train_diag, y_train)

    lr_ranked = y_test.values[np.argsort(-lr.predict_proba(X_test)[:, 1])]
    rf_ranked = y_test.values[np.argsort(-rf.predict_proba(X_test)[:, 1])]
    lr_diag_ranked = y_test.values[np.argsort(-lr_diag.predict_proba(X_test_diag)[:, 1])]
    baseline_ranked = score_baseline(model_df.loc[X_test.index], y_test)

    for k in ks:
        fold_records.append({"fold": fold_num, "k": k,
                              "baseline_rule": precision_at_k(baseline_ranked, k),
                              "logistic_regression": precision_at_k(lr_ranked, k),
                              "lr_with_interaction": precision_at_k(lr_diag_ranked, k),
                              "random_forest": precision_at_k(rf_ranked, k)})

fold_df = pd.DataFrame(fold_records)
print("AFTER — per-fold precision@K:")
print(fold_df.to_string(index=False))

AFTER — per-fold precision@K:
 fold   k  baseline_rule  logistic_regression  lr_with_interaction  random_forest
    0  20          0.500                0.600                0.750          0.550
    0  50          0.480                0.600                0.700          0.660
    0 100          0.470                0.600                0.730          0.690
    0 200          0.540                0.645                0.680          0.670
    1  20          0.500                0.150                0.400          0.550
    1  50          0.540                0.100                0.400          0.580
    1 100          0.590                0.100                0.430          0.600
    1 200          0.605                0.065                0.460          0.565
    2  20          0.300                0.600                0.600          0.600
    2  50          0.280                0.500                0.500          0.500
    2 100          0.290                0.430                0.550  

In [17]:
# Mean ± std across folds
agg = fold_df.groupby("k")[["baseline_rule", "logistic_regression", "lr_with_interaction", "random_forest"]].agg(["mean", "std"])
agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.round(3).reset_index()
print("AFTER — mean ± std across 5 folds:")
print(agg.to_string(index=False))

print("\nBEFORE (single split, for comparison):")
print(before_df.to_string(index=False))

AFTER — mean ± std across 5 folds:
  k  baseline_rule_mean  baseline_rule_std  logistic_regression_mean  logistic_regression_std  lr_with_interaction_mean  lr_with_interaction_std  random_forest_mean  random_forest_std
 20               0.490              0.124                     0.500                    0.203                     0.610                    0.134               0.530              0.057
 50               0.420              0.107                     0.480                    0.228                     0.532                    0.114               0.556              0.065
100               0.412              0.119                     0.432                    0.200                     0.532                    0.119               0.580              0.074
200               0.433              0.131                     0.427                    0.221                     0.505                    0.102               0.577              0.056

BEFORE (single split, for comparison):
  k  

In [18]:
# Bootstrap CI on the RF − baseline gap at K=100 and K=200
rng = np.random.default_rng(42)
n_boot = 10_000

for k in [100, 200]:
    fold_k = fold_df[fold_df["k"] == k]
    diffs = (fold_k["random_forest"] - fold_k["baseline_rule"]).values  # one diff per fold, 5 values

    boot_means = np.array([
        rng.choice(diffs, size=len(diffs), replace=True).mean()
        for _ in range(n_boot)
    ])
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
    contains_zero = ci_low <= 0 <= ci_high

    print(f"K={k}: fold-level RF-baseline diffs = {np.round(diffs, 3)}")
    print(f"  mean diff = {diffs.mean():.3f}, 95% bootstrap CI = [{ci_low:.3f}, {ci_high:.3f}]"
          f"{'  <-- contains zero' if contains_zero else ''}")

K=100: fold-level RF-baseline diffs = [0.22 0.01 0.25 0.23 0.13]
  mean diff = 0.168, 95% bootstrap CI = [0.082, 0.236]
K=200: fold-level RF-baseline diffs = [ 0.13  -0.04   0.21   0.205  0.215]
  mean diff = 0.144, 95% bootstrap CI = [0.045, 0.211]


### Honest Split: What changed, and what it changed

**RF vs. baseline at K=100/200: Reappears.** The single-split gap (0.55 vs 0.51, 0.625 vs 0.51) reappears across all 5 GroupKFold folds, and a bootstrap CI on the fold-level RF-minus-baseline difference stays entirely above zero at both K=100 ([0.082, 0.236]) and K=200 ([0.045, 0.211]).

**RF vs. baseline at K=20/50: Original Result Reversed.** Week 5 read as "the rule wins at short queues" (0.70 vs 0.60 at K=20, 0.60 vs 0.58 at K=50, one split). Under GroupKFold however, RF beats the baseline rule in all 5 folds at K=50 (mean 0.556 vs 0.420), and K=20 is a toss-up — RF wins 3 of 5 folds, loses 2. The single-split baseline@20 of 0.70 was higher than any of the 5 GroupKFold folds' baseline scores (max 0.650), suggesting that split happened to hand the rule an easier test set.

**Rule Stability Clients vs Model.** Rule appears to be less stable across clients than the model, as seen with a fold-to-fold std baseline rule of 0.107–0.131, and a random forest 0.056–0.074. A fixed, hand-built rule swinging more across which clients land in test than a fitted model is counterintuitive and as such is flagged here.

**Wide Unassisted LR Fold-Level Spread.** The fold-level spread of the Unassisted LR appears to be wide. Mean precision sits close to the baseline at three of four Ks, but individual folds range from 0.065 to 0.70 at the same K (fold 1 vs. fold 3 at K=50), so one split can land anywhere in that range.

**Interaction Feature Changes LR Shape.** With `zero_clicks_and_worsened_fh` added, LR's mean precision exceeds RF's at K=20 (0.610 vs 0.530) and nearly matches it at K=50 (0.532 vs 0.556), with RF only pulling clearly ahead at K=100/200. Fold-to-fold std also drops by roughly half at every K (e.g., K=100: 0.200 → 0.119). RF still has the lowest std of all four models at every K, including where its mean is beaten.

**Fold 1 Outlier.** The baseline scores of Fold 1 are the highest there of any fold while both LR variants collapse, and as such will be investigated during the upcoming error checking segment.

**Claims.** Several Week 5 claims — "use the rule for short queues," "LR loses at every K, not narrowly" — don't survive this split unchanged and will have to be rewritten in Section 4.

## 2.5 Error Checking

In [25]:
# Rerun fold loop but keep predictions for use in Section 3
oof_rf = np.full(len(model_df), np.nan)
oof_lr = np.full(len(model_df), np.nan)
oof_lr_diag = np.full(len(model_df), np.nan)
oof_fold = np.full(len(model_df), -1)
fold1_snapshot = None

gkf = GroupKFold(n_splits=5)
for fold_num, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    X_train_diag = model_df.loc[X_train.index, diag_feature_cols].astype(float)
    X_test_diag = model_df.loc[X_test.index, diag_feature_cols].astype(float)

    lr = make_lr(); lr.fit(X_train, y_train)
    rf = make_rf(); rf.fit(X_train, y_train)
    lr_diag = make_lr(); lr_diag.fit(X_train_diag, y_train)

    rf_scores = rf.predict_proba(X_test)[:, 1]
    lr_scores = lr.predict_proba(X_test)[:, 1]
    lr_diag_scores = lr_diag.predict_proba(X_test_diag)[:, 1]

    oof_rf[test_idx] = rf_scores
    oof_lr[test_idx] = lr_scores
    oof_lr_diag[test_idx] = lr_diag_scores
    oof_fold[test_idx] = fold_num

    if fold_num == 1:
        fold1_df = model_df.loc[X_test.index].copy()
        fold1_df["y_true"] = y_test.values
        fold1_df["rf_score"] = rf_scores
        fold1_df["lr_score"] = lr_scores
        fold1_df["lr_diag_score"] = lr_diag_scores
        fold1_df["baseline_score"] = fold1_df["zero_clicks_at_position"] * 2 + fold1_df["position_worsened"] * 1
        fold1_snapshot = fold1_df

model_df["oof_rf_score"] = oof_rf
model_df["oof_lr_score"] = oof_lr
model_df["oof_lr_diag_score"] = oof_lr_diag
model_df["oof_fold"] = oof_fold
model_df["baseline_score"] = model_df["zero_clicks_at_position"] * 2 + model_df["position_worsened"] * 1

print(f"Out-of-fold predictions stored for all {len(model_df)} rows across 5 folds.")
print(f"Fold 1 snapshot: {fold1_snapshot.shape}")

Out-of-fold predictions stored for all 150675 rows across 5 folds.
Fold 1 snapshot: (30086, 23)


In [26]:
# Fold 1 deep dive
print("=== Fold 1 profile vs. overall ===")
print(f"Fold 1 test rows: {len(fold1_snapshot)}, base rate: {fold1_snapshot['y_true'].mean():.3f}")
print(f"Overall base rate: {model_df['is_declining_proxy'].mean():.3f}")

compare_cols = ["avg_position_fh", "log_impressions_fh", "ctr_fh", "position_change", "has_position_trend"]
print("\nFeature means — fold 1 test rows vs. overall:")
print(pd.DataFrame({
    "fold1_mean": fold1_snapshot[compare_cols].mean(),
    "overall_mean": model_df[compare_cols].mean(),
}).round(3))

fold1_top20_baseline = fold1_snapshot.sort_values(
    ["baseline_score", "total_impressions_full"], ascending=[False, False]
).head(20)
fold1_top20_lr = fold1_snapshot.sort_values("lr_score", ascending=False).head(20)

print(f"\nBaseline top-20 in fold 1 — precision: {fold1_top20_baseline['y_true'].mean():.3f}")
print(f"LR (unassisted) top-20 in fold 1 — precision: {fold1_top20_lr['y_true'].mean():.3f}")

fold1_snapshot["lr_rank"] = fold1_snapshot["lr_score"].rank(ascending=False)
mismatch = fold1_top20_baseline.merge(
    fold1_snapshot[["content_hash_id", "lr_rank"]], on="content_hash_id"
)
mismatch = mismatch[mismatch["lr_rank"] > len(fold1_snapshot) / 2]

print(f"\nBaseline top-20 rows LR ranked in the bottom half of fold 1 ({len(mismatch)} rows):")
print(mismatch[["content_hash_id", "avg_position_fh", "log_impressions_fh", "ctr_fh",
                 "position_change", "zero_clicks_at_position", "position_worsened",
                 "baseline_score", "lr_score", "lr_rank", "y_true"]].to_string(index=False))

=== Fold 1 profile vs. overall ===
Fold 1 test rows: 30086, base rate: 0.478
Overall base rate: 0.438

Feature means — fold 1 test rows vs. overall:
                    fold1_mean  overall_mean
avg_position_fh         12.984        16.547
log_impressions_fh       5.276         4.630
ctr_fh                   0.002         0.004
position_change          1.690         0.619
has_position_trend       0.869         0.791

Baseline top-20 in fold 1 — precision: 0.500
LR (unassisted) top-20 in fold 1 — precision: 0.150

Baseline top-20 rows LR ranked in the bottom half of fold 1 (0 rows):
Empty DataFrame
Columns: [content_hash_id, avg_position_fh, log_impressions_fh, ctr_fh, position_change, zero_clicks_at_position, position_worsened, baseline_score, lr_score, lr_rank, y_true]
Index: []


In [27]:
# Fresh Error Test
# Same review as Week 5 cells 23/24, but every row's score came from a model that never saw that row in training — out-of-fold, across all 150K rows, not one 20% split.
oof_top50 = model_df.sort_values("oof_rf_score", ascending=False).head(50)

false_positives = oof_top50[oof_top50["is_declining_proxy"] == 0]
print(f"Out-of-fold RF top-50 (all folds combined): {len(false_positives)} false positives\n")
print(false_positives[["content_hash_id", "avg_position_fh", "total_impressions_fh",
                        "total_clicks_fh", "position_change", "oof_fold", "oof_rf_score"]]
      .head(10).to_string(index=False))

rule_missed = oof_top50[oof_top50["baseline_score"] == 0]
print(f"\nOut-of-fold RF top-50 rows the baseline rule scored 0: {len(rule_missed)}")
print(rule_missed[["content_hash_id", "avg_position_fh", "total_impressions_fh", "total_clicks_fh",
                    "ctr_fh", "position_change", "oof_fold", "oof_rf_score", "is_declining_proxy"]]
      .head(10).to_string(index=False))

Out-of-fold RF top-50 (all folds combined): 24 false positives

         content_hash_id  avg_position_fh  total_impressions_fh  total_clicks_fh  position_change  oof_fold  oof_rf_score
content_bfd481a760fa3064        42.510206                7386.0              9.0        -1.244382         4      0.801034
content_a11bd5663919f057        40.837551               30982.0             25.0        -1.151191         4      0.796557
content_9d72beee0dffbc0f        43.844461                7143.0              3.0        -1.272702         4      0.788793
content_4e48bd81bb37eb4f        47.131404                6850.0              2.0         0.959277         4      0.787350
content_5effb301ded55c21        41.584565                4258.0              4.0         2.851640         4      0.780445
content_19412009bb676d79        45.392643                6381.0              2.0        -1.572244         4      0.775432
content_f5e2cda099b4321c        39.998754                9623.0             15.0  

### 2.5 Error examples — fold 1 deep dive, and a fresh out-of-fold review

**Fold 1 Baseline-wins/LR-collapses Pattern Linked to Distribution Shift.** Fold 1's held-out clients differ from the overall population on every feature that matters to this task: `position_change` mean 1.690 vs. 0.619 overall (far more declining positions), `ctr_fh` mean 0.002 vs. 0.004 (half the click-through rate), and `avg_position_fh` mean 12.98 vs. 16.55 (ranking somewhat better).

This indicates a client subset whose pages rank fine but are sliding and barely getting clicked which is precisely what the baseline rule's two conditions were hand-built to flag, so the rule performs unusually well there (its best fold by far). LR was fit on the other four folds' distribution, and fold 1 is exactly the kind of shift a linear model trained elsewhere would misfire on.

Checking whether LR was actively downgrading the rule's own top picks came back empty — none of the baseline's top-20 rows land in LR's bottom half. Instead, the real story is more diffuse than that. LR's own top-20 most-confident picks are simply a different, weaker set of rows (0.150 precision), not a direct disagreement with the rule.

**Fresh Error Review.** The 24/50 false positive rate roughly matches RF's overall precision@50, so the headline rate isn't alarming. What is worth flagging however is that pooling raw probability scores across all 5 folds and sorting produces a top-50 that's almost entirely `oof_fold == 4` rows.

Fold 4's actual precision@50 (0.520) wasn't RF's best fold — fold 0's was 0.660 — but fold 4's model outputs systematically higher raw scores without being more accurate. That means the pooled "top-50" isn't a clean cross-client sample, instead being skewed toward whichever fold's model happened to be more confident, not more correct. This shows a real limitation of combining out-of-fold scores from 5 separately trained models without calibrating them against each other first, and as such it is named rather than smoothed over.

**Week 5 Failure Mode Repeat.** Most false positives here carry a large-magnitude `position_change`, being mostly big improvements with a couple of big declines. The notebook from Week 5 already flagged "RF over-trusts a large position swing regardless of direction" from the single-split review. As such, seeing the same pattern reappear on an entirely different, honest split makes it a more durable finding than it looked the first time instead of simply being a quirk of one train/test draw.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Four checks against the attack-the-model checklist: timeline boundary, population-selection leakage (does the full-month-dependent join drop rows tied to the outcome window?), a sibling-column test on `log_impressions_fh` (built from the same first-half impression total that sits in the label's own inequality), and a deliberate leak injection to confirm the test harness would actually catch real leakage if it existed.

In [21]:
# Timeline Boundary Check
boundary_check = con.sql(f"""
    SELECT
        MAX(CASE WHEN report_date < '2026-03-16' THEN report_date END) AS max_date_in_fh_features,
        MAX(CASE WHEN report_date < '2026-03-08' THEN report_date END) AS max_date_in_wk1,
        MIN(CASE WHEN report_date >= '2026-03-08' AND report_date < '2026-03-16' THEN report_date END) AS min_date_in_wk2
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
""").df()

print("Timeline boundary check:")
print(boundary_check.to_string(index=False))
print("\nExpect: max_date_in_fh_features = 2026-03-15, max_date_in_wk1 < 2026-03-08,")
print("min_date_in_wk2 >= 2026-03-08. Any date on/after 2026-03-16 here is a bug.")

Timeline boundary check:
max_date_in_fh_features max_date_in_wk1 min_date_in_wk2
             2026-03-15      2026-03-07      2026-03-08

Expect: max_date_in_fh_features = 2026-03-15, max_date_in_wk1 < 2026-03-08,
min_date_in_wk2 >= 2026-03-08. Any date on/after 2026-03-16 here is a bug.


In [22]:
# Population-Selection Leakage
initial_ids = set(pf_fh["content_hash_id"]) & set(label_df["content_hash_id"])
final_ids = set(model_df["content_hash_id"])
dropped_ids = initial_ids - final_ids

dropped_labels = label_df[label_df["content_hash_id"].isin(dropped_ids)]
kept_labels = label_df[label_df["content_hash_id"].isin(final_ids)]

print(f"Rows with first-half features + a label: {len(initial_ids)}")
print(f"Rows that survive into model_df:         {len(final_ids)}")
print(f"Rows dropped along the way:               {len(dropped_ids)}")
print(f"\nDeclining rate among DROPPED rows: {dropped_labels['is_declining_proxy'].mean():.3f}  (n={len(dropped_labels)})")
print(f"Declining rate among KEPT rows:    {kept_labels['is_declining_proxy'].mean():.3f}  (n={len(kept_labels)})")
print(f"\nOverall model_df base rate for reference: {model_df['is_declining_proxy'].mean():.3f}")

Rows with first-half features + a label: 150675
Rows that survive into model_df:         150675
Rows dropped along the way:               0

Declining rate among DROPPED rows: nan  (n=0)
Declining rate among KEPT rows:    0.438  (n=150675)

Overall model_df base rate for reference: 0.438


In [23]:
# Sibling-Column Check
gss_c = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_c, test_idx_c = next(gss_c.split(X, y, groups))
X_train_c, X_test_c = X.iloc[train_idx_c], X.iloc[test_idx_c]
y_train_c, y_test_c = y.iloc[train_idx_c], y.iloc[test_idx_c]

# WITH log_impressions_fh
rf_with = make_rf(); rf_with.fit(X_train_c, y_train_c)
ranked_with = y_test_c.values[np.argsort(-rf_with.predict_proba(X_test_c)[:, 1])]

# WITHOUT log_impressions_fh
cols_without = [c for c in feature_cols if c != "log_impressions_fh"]
rf_without = make_rf(); rf_without.fit(X_train_c[cols_without], y_train_c)
ranked_without = y_test_c.values[np.argsort(-rf_without.predict_proba(X_test_c[cols_without])[:, 1])]

print("Sibling-column check — log_impressions_fh WITH vs WITHOUT (same split):")
print(f"{'k':>5} {'with':>8} {'without':>8}")
for k in ks:
    print(f"{k:>5} {precision_at_k(ranked_with, k):>8.3f} {precision_at_k(ranked_without, k):>8.3f}")
print("\nSkill's rule of thumb: a collapse from ~1.0 toward ~0.7 is the confession.")
print("A small, non-collapsing gap here is reassurance, not proof.")

Sibling-column check — log_impressions_fh WITH vs WITHOUT (same split):
    k     with  without
   20    0.600    0.600
   50    0.580    0.580
  100    0.550    0.520
  200    0.625    0.580

Skill's rule of thumb: a collapse from ~1.0 toward ~0.7 is the confession.
A small, non-collapsing gap here is reassurance, not proof.


In [24]:
# Deliberate Leak Injection
leak_query = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date >= '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_second_half
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

leak_merged = model_df[["content_hash_id"]].merge(leak_query, on="content_hash_id", how="left")
leak_merged["impr_second_half"] = leak_merged["impr_second_half"].fillna(0)
leak_merged["log_impr_second_half"] = np.log1p(leak_merged["impr_second_half"])

X_leaky_full = model_df[feature_cols].astype(float).copy()
X_leaky_full["log_impr_second_half"] = leak_merged["log_impr_second_half"].values

X_train_d, X_test_d = X_leaky_full.iloc[train_idx_c], X_leaky_full.iloc[test_idx_c]

rf_leaky = make_rf(); rf_leaky.fit(X_train_d, y_train_c)
ranked_leaky = y_test_c.values[np.argsort(-rf_leaky.predict_proba(X_test_d)[:, 1])]

print("Leak injection test — same split as check C, one obviously-leaky feature added:")
print(f"{'k':>5} {'clean (check C, with)':>24} {'leaky (+ impr_second_half)':>28}")
for k in ks:
    print(f"{k:>5} {precision_at_k(ranked_with, k):>24.3f} {precision_at_k(ranked_leaky, k):>28.3f}")
print("\nExpect leaky >> clean, approaching 1.0 at low K. If it doesn't move, the harness")
print("isn't sensitive enough to trust the other three checks.")

Leak injection test — same split as check C, one obviously-leaky feature added:
    k    clean (check C, with)   leaky (+ impr_second_half)
   20                    0.600                        1.000
   50                    0.580                        1.000
  100                    0.550                        1.000
  200                    0.625                        1.000

Expect leaky >> clean, approaching 1.0 at low K. If it doesn't move, the harness
isn't sensitive enough to trust the other three checks.


### Leakage audit — attack-checklist results

**Already satisfied by design, no test needed:** every `_fh` feature is built with an explicit `report_date < '2026-03-16'` SQL filter, so the timeline discipline is enforced at the query level, not by convention. The baseline rule is frozen and never fit on data, so it can't leak in the way a trained model could. Splits are grouped by `client_hash_id` (Section 2). Base rate is printed next to every metric throughout. `zero_clicks_and_worsened_fh` — the one product-flag-style feature that exists — is confined to the labeled diagnostic LR and never enters the main baseline/LR/RF comparison.

**A) Timeline boundary — confirmed, not just trusted.** `max_date_in_fh_features` lands on 2026-03-15, the day before the cutoff; the week-1/week-2 split sits exactly at March 8 with no overlap. The SQL filters do what they were written to do.

**B) Population selection — clean, and structurally so.** Zero rows were dropped between the first-half population and the final `model_df`. Full-month impressions are always ≥ first-half impressions, so the full-month gate in `pf_valid` can never exclude a row that already passed the first-half gate — the population is clean because the filter was incapable of doing damage here, not because of a favorable sample.

**C) Sibling-column check on `log_impressions_fh` — reassuring, not a false negative.** With vs. without: identical at K=20/50 (0.600/0.580 both ways), a modest gap at K=100 (0.550 vs 0.520) and K=200 (0.625 vs 0.580). Nowhere near the "~1.0 collapsing to ~0.7" signature the skill describes for a genuinely label-derived feature — consistent with Week 5's permutation importance, where this feature was tied with `ctr_fh` rather than dominating.

**D) Deliberate leak injection — the harness works.** Adding `impr_second_half` directly sends precision to 1.000 at every K. `is_declining_proxy` is a deterministic function of first-half vs. second-half impressions, and the model already carries a first-half impressions proxy — handing it the second-half number lets it reconstruct the label's own comparison almost exactly. This confirms checks B and C had a real chance to catch a problem and didn't fail to notice one.

**Net finding:** unlike Section 2, this section doesn't surface anything that needs walking back. All four checks are clean, and the injection test specifically validates that "clean" is a trustworthy result here rather than an untested assumption.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Two claims from Week 5 didn't hold up once they hit the honest split, and one new issue turned up that didn't appear in Week 5 at all.

### Claim 1 — the queue-length routing rule

The initial claim from Week 5 said to use the baseline rule for short queues and let RF take over for longer ones, based on a single split where the rule beat RF at K=20 and K=50. Here however that split ended up turning out to have been unusually kind to the rule, as its baseline score of 0.70 at K=20 was higher than any of the five honest folds managed, so it looks like an easy test set rather than a typical one.

Once it was run across five client-grouped folds instead, RF came out ahead of the rule at K=50 every single time, and the RF-versus-baseline gap at K=100 and K=200 held up with a bootstrap interval that stayed clearly above zero both times. K=20 is the one length where the original claim is too unstable to make a conclusion, with RF winning three folds and losing two, resulting in no stable direction to report.

Therefore, a rewritten version of the claim would be that at K=50 and above, RF is observed to show a consistent, measured advantage over the baseline rule across five held-out client folds. At K=20 the two are too close to call in either direction, and nobody should be routing queue-length decisions off this data at that length until there's more to go on.

### Claim 2 — logistic regression "loses at every K"

The unassisted LR model looked uniformly bad in Week 5, scoring below the baseline at every K and below the base rate too. However, that picture doesn't survive five folds. LR's average precision actually sits close to the baseline at three of the four K values, only clearly behind at K=200. What made it look catastrophic before was landing on one particular fold where it fell to 0.065 at K=200, dragging down the overall impression even though the other folds looked fine.

The standard deviation across folds is larger than LR's own mean at several K values, which means that the headline finding is actually that this model isn't consistently weak, it's unpredictable, and those are different problems with different fixes. When it was handed the baseline rule's own interaction feature as a diagnostic, that instability roughly cut in half at every K, and LR became genuinely competitive with RF at the shorter queue lengths.

Therefore, a rewritten version of the claim would be that when unassisted LR's precision swings a lot depending on which clients it's tested against, competitive with the baseline most of the time but far behind on one fold, which makes it unreliable rather than reliably weak. Give it the rule's interaction feature and that swing drops by about half, putting it in real contention with RF at shorter queues.

### The new issue

This issue doesn't actually have a Week 5 version to correct, since it only showed up once predictions from all five folds got pooled together for the fresh error review. Combining raw probability scores from five separately trained models isn't safe without calibrating them against each other first, with fold 4 standing as evidence. Fold 4's model produced the highest raw scores of any fold once everything was pooled, despite not actually being the most accurate fold. Therefore it should be noted going forward that any future review pooling scores across folds needs to account for that before treating the result as a clean, representative sample.

In [20]:
# Section 4 involves rewriting claims using information gathered in previous cells — no query needed

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.